## Contexto biotecnológico y fisicoquímico del dataset (Biomass Pyrolysis)

Este dataset describió **pirólisis de biomasa**, una conversión termoquímica donde una matriz lignocelulósica (principalmente celulosa, hemicelulosa y lignina) se calentó en ausencia de oxígeno para redistribuir masa y energía entre tres familias de productos: **sólido carbonoso (char)**, **líquido condensable (bio-oil + agua)** y **gas no condensable** (CO2, CO, H2, CH4 y otros C1–C3). La lógica causal fue: **(composición/estructura inicial)** + **(transferencia de calor y residencia de vapores)** → competencia entre **desvolatilización**, **craqueo secundario**, **repolimerización/aromatización** y **gasificación** del char.

### Significado químico de las variables de entrada

**Proximate analysis (M, Ash, VM, FC).**  
**M (humedad, %)** afectó el balance térmico por el calor latente de evaporación y, además, favoreció química secundaria en fase vapor; en general tendió a desplazar rendimiento desde líquido orgánico hacia gas y hacia una fracción líquida más acuosa.  
**Ash (cenizas, %)** representó minerales (K, Na, Ca, Mg, Si, P, etc.) que actuaron como catalizadores heterogéneos, promoviendo craqueo y desoxigenación de vapores y modificando la estructura del char.  
**VM (materia volátil, %)** fue la fracción que sale como vapores al calentar; mayor VM suele implicar mayor potencial de líquido/gas.  
**FC (carbono fijo, %)** aproximó la fracción que permanece como matriz sólida carbonosa; mayor FC tendió a predecir mayor rendimiento de sólido.

**Ultimate analysis (C, H, O, N).**  
Estas variables resumieron el grado de reducción/oxidación del feedstock y, por ende, la propensión a formar gas vs líquido. Un **O** alto implicó abundancia de grupos oxigenados en la biomasa (hidroxilo, -OH; éter, -O-; carbonilo, C=O; carboxilo, -COOH; metoxi, -OCH3), que durante pirólisis generaron H2O, CO y CO2 y aumentaron la fracción de compuestos oxigenados del bio-oil. En términos de tendencia, mayor **O/C** suele favorecer más gas (CO/CO2) y un líquido más polar/oxigenado; mayor **H/C** suele asociarse a productos líquidos más energéticos. **N** se vinculó con especies nitrogenadas en vapores (por ejemplo amina, -NH2; nitrilo, -C≡N) y con implicancias ambientales/upgrading.

### Variables de proceso (ingeniería de reacción)

**PS (tamaño de partícula, mm)** controló transferencia de calor y gradientes internos: partículas grandes calentaron más lento en el centro, extendiendo el tiempo efectivo en condiciones que favorecieron carbonización y, por tanto, char; partículas pequeñas se aproximaron a pirólisis rápida y favorecieron líquido si la condensación fue eficiente.  
**FT (temperatura final, °C)** fue el driver cinético principal: a bajas temperaturas se favoreció char; al aumentar FT creció la liberación de volátiles y el líquido puede aumentar hasta un máximo; a FT altas dominaron craqueo secundario y gasificación, aumentando gas.  
**HR (tasa de calentamiento, °C/min)** separó regímenes “lento” vs “rápido”: HR alta generó vapores rápidamente y tendió a maximizar líquido (si se redujo residencia y se enfrió/condensó rápido); HR baja favoreció repolimerización/aromatización y char.  
**FR (caudal de gas inerte, ml/min)** moduló el tiempo de residencia de vapores: FR alto arrastró vapores fuera de la zona caliente, reduciendo craqueo y aumentando líquido; FR bajo aumentó residencia, favoreciendo craqueo a gases ligeros y deposición de coque secundario (que suma a sólido).

### Variables objetivo (rendimientos de fase)

**Solid phase / Liquid phase / Gas phase (%, masa)** fueron rendimientos de cada fracción. Conceptualmente, “solid” incluyó char + cenizas; “liquid” incluyó todo lo condensable (bio-oil orgánico + agua); “gas” fue lo no condensable. Un chequeo mínimo fue que **las tres fases cierren ~100%**; desviaciones sistemáticas sugieren bases diferentes (base húmeda vs base seca) o heterogeneidad experimental entre fuentes.

### Implicancias para el EDA y el modelado 
Este dataset fue un problema **multisalida** con tres rendimientos acoplados por conservación de masa: si aumenta una fracción, alguna otra debe disminuir. Para evitar incoherencias, una estrategia robusta fue modelar **dos fases** y obtener la tercera por cierre a 100%, o usar un enfoque que respete explícitamente la restricción. Antes de interpretar correlaciones, fue crítico verificar la **base de las variables** (especialmente M y los rendimientos) y auditar el cierre de masa; mezclar “as received” con “dry basis” puede crear correlaciones espurias que no reflejan química real.

In [11]:
import pandas as pd
import numpy as np
RAW_PATH = r"C:\Users\santi\Documents\Pyrolisis\Data\pyrolysis.csv"
df = pd.read_csv(RAW_PATH)

In [12]:
# Mostrar información básica del dataset
print("Dimensión (filas, columnas):", df.shape)
print("Columnas:", df.columns.tolist())
df.head()
# Verificar tipos de datos
print(df.dtypes)
# Verificar valores faltantes
print("Valores faltantes por columna:\n", df.isnull().sum())


Dimensión (filas, columnas): (751, 17)
Columnas: ['Index', 'Biomass species', 'M', 'Ash ', 'VM', 'FC', 'C', 'H', 'O', 'N', 'PS', 'FT', 'HR', 'FR', 'Solid phase', 'Liquid phase', 'Gas phase']
Index                int64
Biomass species        str
M                  float64
Ash                float64
VM                 float64
FC                 float64
C                  float64
H                  float64
O                  float64
N                  float64
PS                     str
FT                   int64
HR                 float64
FR                 float64
Solid phase            str
Liquid phase           str
Gas phase              str
dtype: object
Valores faltantes por columna:
 Index                0
Biomass species      0
M                  119
Ash                 18
VM                  68
FC                  68
C                    3
H                    3
O                    3
N                   30
PS                 101
FT                   0
HR                 110
FR   

In [13]:
# Porcentaje de valores faltantes por columna
#Antes ya habiamos dicho que era cero (0) pero para que quede en porcentaje  --- IGNORE ---
df.isnull().mean()*100
print (df.isnull().mean()*100)

Index               0.000000
Biomass species     0.000000
M                  15.845539
Ash                 2.396804
VM                  9.054594
FC                  9.054594
C                   0.399467
H                   0.399467
O                   0.399467
N                   3.994674
PS                 13.448735
FT                  0.000000
HR                 14.647137
FR                 24.101198
Solid phase         1.198402
Liquid phase        4.793609
Gas phase           4.793609
dtype: float64


In [14]:
units_df = pd.read_csv(r'C:\Users\santi\Documents\Pyrolisis\Data\pyrolysis_column_units.csv')
print(units_df.head())
print(units_df.columns.tolist())

   Index  Biomass species  M Ash  VM FC  C  H  O  N  PS FT HR      FR  \
0    NaN              NaN  %    %  %  %  %  %  %  %  mm  ℃  ℃  mL/min   

  Solid phase Liquid phase Gas phase  
0           %            %         %  
['Index', 'Biomass species', 'M', 'Ash ', 'VM', 'FC', 'C', 'H', 'O', 'N', 'PS', 'FT', 'HR', 'FR', 'Solid phase', 'Liquid phase', 'Gas phase']


In [15]:
#1. Se esperaba de los siguientes datos un número, no un srtring, por lo que se debe revisar el dataset para entender por qué hay valores no numéricos en estas columnas. Es posible que haya errores de formato, unidades mezcladas o datos categóricos que necesitan ser convertidos a numéricos.
print(df['Solid phase'].unique()[:20])
print(df['Liquid phase'].unique()[:20])
print(df['Gas phase'].unique()[:20])

<StringArray>
[   '35', '31.75',  '30.2',  '28.6', '32.85', '30.14',  '28.4', '26.28',
 '52.57', '39.26', '28.82', '26.53', '24.84',  '24.1', '23.33', '56.69',
 '47.28', '46.56',  '57.7',  '47.2']
Length: 20, dtype: str
<StringArray>
[ '44.9', '41.25',  '40.6', '36.36', '54.38', '55.56', '54.75', '53.22',
 '28.48', '35.38', '33.56', '33.33', '31.64', '31.02',  '29.3', '25.04',
 '30.66', '30.19',  '20.8',  '22.2']
Length: 20, dtype: str
<StringArray>
[ '20.1',    '27',  '29.2', '35.04', '12.77',  '14.3', '16.85',  '20.5',
 '18.95', '25.34', '37.61', '39.73',  '42.9',  '44.1',  '45.8',  '14.8',
 '21.16', '21.89',  '21.7',  '29.7']
Length: 20, dtype: str


# Solid/Liquid/Gas phas los valores son números pero pandas los leyó como texto

In [16]:
#2. Para convertir estas columnas a numéricas, se puede usar la función pd.to_numeric() de pandas, que intentará convertir los valores a números y asignará NaN a aquellos que no puedan convertirse. Esto permitirá realizar análisis estadísticos y visualizaciones posteriormente.
df['Solid phase'] = pd.to_numeric(df['Solid phase'], errors='coerce')
df['Liquid phase'] = pd.to_numeric(df['Liquid phase'], errors='coerce')
df['Gas phase'] = pd.to_numeric(df['Gas phase'], errors='coerce')

In [17]:
#1. La columna "PS" también parece contener valores no numéricos, por lo que se debe revisar su contenido para entender qué tipo de datos contiene y cómo se pueden convertir a un formato adecuado para el análisis.
print(df['PS'].unique()[:20])


<StringArray>
[    '0.5',    '0.55',     '0.3',     '0.1',    '0.65',     '0.4',       nan,
    '0.43',    '0.32',    '0.75', '0.5-0.6',    '0.25',    '0.38',     '0.8',
    '1.35',    '0.83',    '1.25',   '11.25',    '0.21',     '0.2']
Length: 20, dtype: str


In [18]:
#Dentro de los números había rangos representados con un guion "-", por lo que se puede analizar la frecuencia de estos valores para entender mejor su contenido y decidir cómo manejarlos en el análisis posterior.

In [19]:
#2. Para entender mejor el contenido de la columna "PS", se puede analizar la frecuencia de los valores que contienen un guion "-", lo que podría indicar rangos o categorías específicas. Esto ayudará a decidir cómo manejar estos datos en el análisis posterior.
mascara_rango = df['PS'].str.contains('-', na=False)
print(df['PS'][mascara_rango].value_counts())
print("Total:", mascara_rango.sum())

PS
0.5-0.6    14
Name: count, dtype: int64
Total: 14


PS contiene 14 valores en formato rango ('0.5-0.6'). Representan menos del 2% del dataset. Se reemplazaron por el promedio del rango para permitir la conversión a numérico. No se creó variable indicadora por insuficiente representación.

In [20]:
#3. Para convertir los valores de la columna "PS" a numéricos, se puede crear una función personalizada que maneje los casos de rangos (valores con guion) y los convierta a un valor numérico representativo, como el promedio de los extremos del rango. Para los valores que no contienen un guion, simplemente se convertirán a float.
def convertir_ps(valor):
    if pd.isna(valor):
        return np.nan
    if '-' in str(valor):
        partes = str(valor).split('-')
        return (float(partes[0]) + float(partes[1])) / 2
    return float(valor)

df['PS'] = df['PS'].apply(convertir_ps)

In [21]:
# Verificar nuevamente los tipos de datos y valores faltantes después de la conversión
print(df.isnull().sum().to_string())


Index                0
Biomass species      0
M                  119
Ash                 18
VM                  68
FC                  68
C                    3
H                    3
O                    3
N                   30
PS                 101
FT                   0
HR                 110
FR                 181
Solid phase         12
Liquid phase        39
Gas phase           39


In [22]:
# Mostrar las primeras filas del dataframe para verificar los cambios
print(df.head())

   Index            Biomass species      M  Ash      VM     FC      C     H  \
0      1  Jerusalem artichoke stick  15.76  3.34  67.40  13.50  45.36  6.11   
1      2  Jerusalem artichoke stick  15.76  3.34  67.40  13.50  45.36  6.11   
2      3  Jerusalem artichoke stick  15.76  3.34  67.40  13.50  45.36  6.11   
3      4  Jerusalem artichoke stick  15.76  3.34  67.40  13.50  45.36  6.11   
4      5                       reed   5.89  8.47  72.12  13.52  42.78  5.17   

       O     N   PS   FT    HR     FR  Solid phase  Liquid phase  Gas phase  
0  47.26  0.75  0.5  550  10.0  100.0        35.00         44.90      20.10  
1  47.26  0.75  0.5  650  10.0  100.0        31.75         41.25      27.00  
2  47.26  0.75  0.5  750  10.0  100.0        30.20         40.60      29.20  
3  47.26  0.75  0.5  850  10.0  100.0        28.60         36.36      35.04  
4  50.51  1.33  0.5  550  10.0  100.0        32.85         54.38      12.77  


In [23]:
# Porcentaje de valores faltantes por columna
#Antes ya habiamos dicho que era cero (0) pero para que quede en porcentaje  --- IGNORE ---
df.isnull().mean()*100
print (df.isnull().mean()*100)

Index               0.000000
Biomass species     0.000000
M                  15.845539
Ash                 2.396804
VM                  9.054594
FC                  9.054594
C                   0.399467
H                   0.399467
O                   0.399467
N                   3.994674
PS                 13.448735
FT                  0.000000
HR                 14.647137
FR                 24.101198
Solid phase         1.597870
Liquid phase        5.193076
Gas phase           5.193076
dtype: float64


# Solid phase pasó de 9 a 12 nulos, Liquid phase y Gas phase de 36 a 39. Eso significa que había 3 filas con valores que no eran ni números ni rangos, algo como texto o símbolos. El errors='coerce' los convirtió a NaN automáticamente. Eso es exactamente para lo que sirve ese parámetro.

In [24]:
# ¿Los nulos de FR coinciden con nulos de HR?
    # Si ambos son nulos al mismo tiempo, podría indicar que hay un problema específico con la recolección de datos para esas filas, o que estas variables están relacionadas de alguna manera. Si no coinciden, entonces los nulos podrían ser independientes y requerir un tratamiento diferente.
print(df[['HR','FR']].isnull().sum())
print("Nulos en ambas a la vez:", df[['HR','FR']].isnull().all(axis=1).sum())

# ¿Los nulos de FR se concentran en alguna especie?
    # Si los nulos de FR se concentran en ciertas especies, podría indicar que la medición de FR no se realizó para esas especies específicas, o que hubo un problema con la recolección de datos para esas filas. Esto podría requerir un tratamiento específico para esos casos, como imputar valores basados en otras características o excluir esas filas del análisis.
print(df[df['FR'].isnull()]['Biomass species'].value_counts().head(10))

# ¿La temperatura FT es diferente cuando FR es nulo?
    #  Si la temperatura FT es significativamente diferente cuando FR es nulo, podría indicar que hay una relación entre estas dos variables, o que los nulos de FR están asociados con ciertas condiciones de temperatura. Esto podría requerir un análisis más detallado para entender la naturaleza de esta relación y cómo manejar los nulos en el análisis posterior.
print("FT cuando FR es nulo:", df[df['FR'].isnull()]['FT'].describe())
print("FT cuando FR no es nulo:", df[df['FR'].notna()]['FT'].describe())

HR    110
FR    181
dtype: int64
Nulos en ambas a la vez: 62
Biomass species
Lemon grass                 12
Cherry seed                 12
wheat straw , oat straw     12
spirulina                   12
rapeseed                    12
Olive shell                 11
Corn straw                  10
rice straw                  10
Carpinus betulus L.          9
tea dust                     9
Name: count, dtype: int64
FT cuando FR es nulo: count     181.000000
mean      563.259669
std       182.076400
min       300.000000
25%       450.000000
50%       500.000000
75%       600.000000
max      1250.000000
Name: FT, dtype: float64
FT cuando FR no es nulo: count    570.000000
mean     511.649123
std      106.057731
min      300.000000
25%      450.000000
50%      500.000000
75%      550.000000
max      900.000000
Name: FT, dtype: float64


In [25]:
# ¿Los nulos de M se concentran en alguna especie?
    # Si los nulos de M se concentran en ciertas especies, podría indicar que la medición de M no se realizó para esas especies específicas, o que hubo un problema con la recolección de datos para esas filas. Esto podría requerir un tratamiento específico para esos casos, como imputar valores basados en otras características o excluir esas filas del análisis.
nulos_por_especie = df[df['M'].isnull()]['Biomass species'].value_counts()
print(nulos_por_especie.head(10))

# ¿Cuántas especies tienen TODOS sus valores de M nulos?
    # Si hay especies que tienen todos sus valores de M nulos, esto podría indicar un problema específico con la recolección de datos para esas especies, o que la variable M no se midió para esas especies en particular. Esto podría requerir un tratamiento especial, como excluir esas especies del análisis o imputar valores basados en otras características.
total_por_especie = df.groupby('Biomass species')['M'].count()
print(total_por_especie[total_por_especie == 0])

Biomass species
paulownia wood             18
F. orientalis L. plants    18
Lemon grass                12
Olive shell                11
Corn straw                 10
tea dust                    9
Moso bamboo                 8
Microalgae                  6
Cigarette rod               5
straw                       5
Name: count, dtype: int64
Biomass species
Cigarette rod              0
Degreasing cake            0
Dwarf palm                 0
F. orientalis L. plants    0
Green bristlegrass         0
Lemon grass                0
Little blue star           0
Long leaved pine leaves    0
Marsh Bay                  0
Microalgae                 0
Mistletoe                  0
Moso bamboo                0
Myrtle wax                 0
Olive shell                0
Pinus longifolia litter    0
Sawn palm                  0
Smooth Holly               0
Sparkle Berry              0
Stumbling shrub            0
Tea substitute Holly       0
Water oak                  0
straw                      0
tea 

In [26]:
# Verificar que las columnas de fases sumen aproximadamente 100
df['suma_fases'] = df['Solid phase'] + df['Liquid phase'] + df['Gas phase']
print(df['suma_fases'].describe())

# ¿Cuántas filas NO suman aproximadamente 100?
fuera_de_rango = df[(df['suma_fases'] < 99) | (df['suma_fases'] > 101)]
print("Filas que no suman 100 ±1%:", len(fuera_de_rango))
print(fuera_de_rango[['Solid phase', 'Liquid phase', 'Gas phase', 'suma_fases']].head(10))

count    712.000000
mean      96.416882
std        8.006419
min       55.070000
25%       97.980000
50%      100.000000
75%      100.000000
max      118.300000
Name: suma_fases, dtype: float64
Filas que no suman 100 ±1%: 241
    Solid phase  Liquid phase  Gas phase  suma_fases
14        23.33         29.30      45.80       98.43
15        56.69         25.04      14.80       96.53
17        46.56         30.19      21.89       98.64
22        32.60         20.10      46.20       98.90
34        23.12         34.35      40.84       98.31
35        21.48         33.76      42.10       97.34
41        64.87         13.89      23.30      102.06
42        20.99         26.45      55.00      102.44
43        20.78         22.34      60.71      103.83
44        21.10         30.51      52.69      104.30


In [27]:
# ¿Las filas que no suman 100 tienen más nulos en las fases?
fuera_de_rango = df[(df['suma_fases'] < 99) | (df['suma_fases'] > 101)]
print(fuera_de_rango[['Solid phase', 'Liquid phase', 'Gas phase', 'suma_fases']].isnull().sum())

# ¿Las que suman menos de 90 o más de 110 son casos extremos?
extremos = df[(df['suma_fases'] < 90) | (df['suma_fases'] > 110)]
print(f"Casos extremos (fuera de 90-110): {len(extremos)}")
print(extremos[['Biomass species', 'Solid phase', 'Liquid phase', 'Gas phase', 'suma_fases']])

Solid phase     0
Liquid phase    0
Gas phase       0
suma_fases      0
dtype: int64
Casos extremos (fuera de 90-110): 107
                 Biomass species  Solid phase  Liquid phase  Gas phase  \
123          Carpinus betulus L.        40.37         18.63      13.68   
124          Carpinus betulus L.        36.69         21.03      15.75   
125          Carpinus betulus L.        35.25         22.15      16.23   
126          Carpinus betulus L.        34.45         22.15      16.55   
127          Carpinus betulus L.        33.33         20.55      18.47   
..                           ...          ...           ...        ...   
675  White oak and sweetgum logs        18.30         56.20      43.80   
739              Eucalyptus wood        25.17         32.92      31.84   
740              Eucalyptus wood        25.38         36.73      27.36   
741                         corn        11.90         59.90      10.00   
745                         corn        30.10         64.10    

In [28]:
# ¿Cuántas especies únicas hay en los extremos?
print(extremos['Biomass species'].value_counts())

Biomass species
Grape pomace                     16
rice straw                       14
Rhinoceros sawdust               13
Carpinus betulus L.              12
wheat straw , oat straw          12
Corncob                          11
P. glabra and M. ferrea seeds     8
rapeseed                          5
Apricot shell                     4
Switchgrass                       3
White oak and sweetgum logs       2
Eucalyptus wood                   2
corn                              2
grass                             1
OPMF,PF                           1
Beech                             1
Name: count, dtype: int64


#conclusión para @02-cleaning

El dataset quedó caracterizado como una tabla de **751 filas y 17 columnas**, con una combinación típica de variables de **composición del feedstock** (M, Ash, VM, FC, C, H, O, N), **condiciones de proceso** (PS, FT, HR, FR) y **rendimientos** (Solid/Liquid/Gas). El primer hallazgo relevante fue que existieron **problemas de tipado y formato**: los rendimientos de fase habían sido leídos como texto y se convirtieron a numérico con coerción, lo que expuso un pequeño subconjunto de entradas no parseables que pasó a NaN. En PS aparecieron rangos del tipo “0.5–0.6” (14 filas, <2%); se reemplazaron por el promedio del rango para preservar continuidad numérica sin introducir una variable indicadora porque la representación fue marginal.

La **calidad de datos** estuvo dominada por faltantes estructurados más que por ruido aleatorio. FR presentó el mayor faltante (≈24.1%), seguido por M (≈15.8%) y HR (≈14.6%); además, HR y FR mostraron co-ocurrencia de nulos (62 filas), lo que sugirió ausencia conjunta de metadatos de operación en un subconjunto de experimentos. Ese patrón no fue neutro: cuando FR fue nulo, FT mostró una distribución desplazada hacia valores más altos (media ~563 °C) respecto de los casos con FR presente (media ~512 °C), compatible con heterogeneidad experimental (p. ej., fuentes distintas o campañas con registro incompleto) y con el riesgo de sesgo si se imputó sin estratificar.

El chequeo fisicoquímico más importante fue el **cierre de masa** de los rendimientos. En 712 filas con las tres fases disponibles, la suma Solid+Liquid+Gas tuvo mediana 100 pero una dispersión apreciable (media ~96.4; mínimo ~55.1; máximo ~118.3). Un total de 241 filas quedó fuera de 100±1%, y 107 casos fueron extremos al exceder el rango 90–110, sin que ello se explicara por NaN en las fases (fueron valores numéricos cohercibles). Este patrón fue consistente con **mezcla de bases** (base seca vs “as received”, inclusión/exclusión de agua en “liquid”, o definiciones operativas distintas de “bio-oil”) y/o con redondeos/reportes heterogéneos entre especies/estudios, por lo que el modelado multisalida requirió tratar explícitamente la restricción de cierre o aceptar que el target incorporó ruido de medición/convención.

En síntesis, el EDA mostró que el dataset fue utilizable para análisis y modelado, pero su limitación principal no estuvo en la cantidad de filas sino en la **heterogeneidad de registro** (faltantes correlacionados) y en la **inconsistencia parcial del balance de masa**. En el pipeline posterior, las decisiones críticas fueron separar “limpieza de formato” de “calidad experimental”, auditar e idealmente armonizar la base de rendimientos, y definir una estrategia de tratamiento de nulos que no borre estructura (especialmente en FR/HR/M) ni introduzca incoherencias termodinámicas por imputación indiscriminada.